# Mistral na T4: FP16, NF4 e API real
Sessão privada. Confirme a cota no Kaggle, selecione T4 x2 e habilite internet. A execução expõe só a GPU 0. O pacote de resultados permanece em /kaggle/working.

In [ ]:
import json
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path
from time import monotonic

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
BUDGET_MIN = 35
deadline = monotonic() + BUDGET_MIN * 60
working = Path("/kaggle/working")
out = working / "mistral-t4-results"
out.mkdir()
repo = Path("/tmp/partner-copilot-benchmark")
commit = "8bf96afb41b386c9a0bb750ce3235b5cfbe46461"
protocol_commit = "5c65ba4dfa50db4a734c6be3187b4f244e053f4e"


def journal(**record):
    with (out / "session.jsonl").open("a", encoding="utf-8") as file:
        file.write(json.dumps(record) + "\n")
        file.flush()
        os.fsync(file.fileno())


def run(command, *, cap=None):
    seconds = deadline - monotonic()
    if seconds <= 0:
        raise TimeoutError("Session budget exhausted")
    with subprocess.Popen(command, start_new_session=True) as child:
        try:
            returncode = child.wait(timeout=min(seconds, cap or seconds))
        except subprocess.TimeoutExpired:
            os.killpg(child.pid, signal.SIGKILL)
            child.wait()
            raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)


journal(
    event="session",
    base_commit=commit,
    protocol_commit=protocol_commit,
    budget_minutes=BUDGET_MIN,
    quota_minutes_before=88,
    visible_devices="0",
    python=sys.version.split()[0],
    gpu=subprocess.check_output(
        [
            "nvidia-smi",
            "-i",
            "0",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        text=True,
    ).strip(),
)
run(["git", "clone", "https://github.com/JoaoNordemann/partner-copilot.git", str(repo)])
os.chdir(repo)
run(["git", "checkout", "--detach", commit])
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "httpx",
        "transformers==5.20.2",
        "bitsandbytes==0.50.2",
        "accelerate==1.15.0",
        "chromadb==1.5.9",
        "sentence-transformers==5.7.0",
    ]
)

In [ ]:
for name, source in {
    "benchmark_gpu_precision.py": '"""Run each precision in a fresh Kaggle worker and journal each observation."""\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom time import perf_counter\n\nROOT = Path("reports/final-v3")\nMODEL = "mistralai/Mistral-7B-Instruct-v0.2"\n\n\ndef append(path, **row):\n    with path.open("a", encoding="utf-8") as stream:\n        stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n\n\ndef cases():\n    hashes = dict(line.split()[::-1] for line in (ROOT / "SHA256SUMS").read_text().splitlines())\n    result = {}\n    for group, file in (\n        ("limpo_original", "generation-v3-new.jsonl"),\n        ("contaminado", "generation-v3-contaminated.jsonl"),\n    ):\n        data = (ROOT / file).read_bytes().replace(b"\\r\\n", b"\\n")\n        if hashlib.sha256(data).hexdigest() != hashes[file]:\n            raise ValueError(f"Hash mismatch: {file}")\n        result[group] = [r for line in data.splitlines() if (r := json.loads(line))["llm_called"]]\n    if [len(v) for v in result.values()] != [9, 27]:\n        raise ValueError("Unexpected cohort sizes")\n    return result\n\n\ndef gpu_worker(mode, snapshot, path):\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n    from partner_copilot.generation.contracts import GenerationContractError\n    from partner_copilot.generation.output import parse_generation\n    from partner_copilot.generation.output_v3 import decode_output\n    from partner_copilot.schemas import RetrievedDoc, Review\n\n    if torch.cuda.device_count() != 1 or torch.cuda.get_device_name(0) != "Tesla T4":\n        raise RuntimeError("Exactly one visible Tesla T4 is required")\n\n    def memory():\n        free, total = torch.cuda.mem_get_info(0)\n        return {\n            "free_bytes": free,\n            "total_bytes": total,\n            "allocated_bytes": torch.cuda.memory_allocated(0),\n            "reserved_bytes": torch.cuda.memory_reserved(0),\n            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),\n            "peak_reserved_bytes": torch.cuda.max_memory_reserved(0),\n        }\n\n    append(\n        path,\n        event="environment",\n        mode=mode,\n        gpu=torch.cuda.get_device_name(0),\n        cuda=torch.version.cuda,\n        revision=snapshot.name,\n        python=sys.version.split()[0],\n        memory=memory(),\n        commit=subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),\n        driver=subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],\n            text=True,\n        ).splitlines()[0],\n        packages={\n            name: version(name)\n            for name in ("torch", "transformers", "accelerate", "bitsandbytes", "chromadb")\n        },\n    )\n    groups = cases()\n    stage, identifier = "load", None\n    try:\n        torch.cuda.reset_peak_memory_stats(0)\n        started = perf_counter()\n        tokenizer = AutoTokenizer.from_pretrained(snapshot, local_files_only=True)\n        kwargs = {}\n        if mode == "nf4":\n            kwargs["quantization_config"] = BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type="nf4",\n                bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n            )\n        model = AutoModelForCausalLM.from_pretrained(\n            snapshot,\n            local_files_only=True,\n            dtype=torch.float16,\n            device_map={"": 0},\n            attn_implementation="sdpa",\n            **kwargs,\n        ).eval()\n        torch.cuda.synchronize(0)\n        append(path, event="load", seconds=perf_counter() - started, memory=memory())\n        for group, rows in (("warmup", groups["contaminado"][:2]), *groups.items()):\n            stage = group\n            torch.cuda.reset_peak_memory_stats(0)\n            for row in rows:\n                identifier = row["question_id"]\n                torch.cuda.synchronize(0)\n                started = perf_counter()\n                chat = tokenizer.apply_chat_template(\n                    [{"role": "user", "content": row["rendered_prompt"]}],\n                    tokenize=False,\n                    add_generation_prompt=True,\n                )\n                inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to("cuda:0")\n                prompt_tokens = inputs["input_ids"].shape[-1]\n                if prompt_tokens + 512 + 256 > min(\n                    model.config.max_position_embeddings, tokenizer.model_max_length\n                ):\n                    raise ValueError("Prompt exceeds frozen context budget")\n                torch.manual_seed(42)\n                torch.cuda.synchronize(0)\n                generation_start = perf_counter()\n                with torch.inference_mode():\n                    output = model.generate(\n                        **inputs,\n                        max_new_tokens=512,\n                        do_sample=False,\n                        pad_token_id=tokenizer.eos_token_id,\n                        use_cache=True,\n                    )[0, prompt_tokens:]\n                torch.cuda.synchronize(0)\n                generation_seconds = perf_counter() - generation_start\n                raw = tokenizer.decode(output, skip_special_tokens=True)\n                latency = perf_counter() - started\n                docs = [\n                    RetrievedDoc(\n                        review=Review.model_construct(id=d["review_id"], text=d["text"]),\n                        score=d["score"],\n                    )\n                    for d in row["context"]\n                ]\n                parsed, error = None, None\n                try:\n                    parse_generation(raw, docs, parser_version="parser-v3")\n                    parsed = decode_output(raw).value.model_dump(mode="json")\n                except GenerationContractError as exc:\n                    error = type(exc).__name__\n                append(\n                    path,\n                    event="generation",\n                    group=group,\n                    id=identifier,\n                    raw=raw,\n                    accepted=error is None,\n                    structured=parsed,\n                    parser_error=error,\n                    seconds=latency,\n                    generated_tokens=len(output),\n                    generation_seconds=generation_seconds,\n                    prompt_tokens=prompt_tokens,\n                    memory=memory(),\n                )\n    except torch.OutOfMemoryError as exc:\n        append(path, event="oom", stage=stage, id=identifier, error=str(exc), memory=memory())\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=("nf4", "fp16"))\n    parser.add_argument("snapshot", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    os.environ["CUDA_VISIBLE_DEVICES"] = "0"\n    args.output.touch(exist_ok=False)\n    gpu_worker(args.mode, args.snapshot, args.output)\n\n\nif __name__ == "__main__":\n    main()\n',  # noqa: E501
    "benchmark_gpu_api.py": '"""Measure the real API through loopback HTTP and persist each response."""\n\nimport argparse\nimport os\nimport subprocess\nimport sys\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom time import perf_counter, sleep\n\nfrom benchmark_gpu_precision import append, cases\n\n\ndef serve(log_path):\n    import uvicorn\n\n    from partner_copilot.api import create_app\n    from partner_copilot.api_runtime import Resources\n\n    class ObservedResources(Resources):\n        def start(self):\n            super().start()\n            if self.llm is not None:\n                original = self.llm.generate\n\n                def timed(prompt):\n                    start = perf_counter()\n                    try:\n                        return original(prompt)\n                    finally:\n                        append(log_path, event="llm_interval", started=start, ended=perf_counter())\n\n                self.llm.generate = timed\n\n    uvicorn.run(create_app(resources_factory=ObservedResources), host="127.0.0.1", port=8765)\n\n\ndef measure(path):\n    import httpx\n\n    endpoint = "http://127.0.0.1:8765"\n    rows = {r["question_id"]: r for r in cases()["contaminado"]}\n    trace = path.with_name("api-generation-intervals.jsonl")\n    server_log = path.with_name("api-server.log")\n    append(path, event="configuration", api_request_timeout_s=60, generation_slots=1)\n    with (\n        server_log.open("x", encoding="utf-8") as log,\n        subprocess.Popen(\n            [sys.executable, __file__, "serve", str(trace)], stdout=log, stderr=log\n        ) as server,\n    ):\n        try:\n            deadline = perf_counter() + 240\n            while perf_counter() < deadline and server.poll() is None:\n                try:\n                    response = httpx.get(endpoint + "/health", timeout=2)\n                    append(path, event="health", status=response.status_code, body=response.json())\n                    if response.is_success:\n                        break\n                    raise RuntimeError("API startup reported an unhealthy component")\n                except httpx.TransportError:\n                    sleep(1)\n            else:\n                raise RuntimeError("API did not become healthy")\n\n            def request(identifier, group, barrier=None):\n                row = rows[identifier]\n                scope = row["filtros"].copy()\n                body = {\n                    "restaurant_id": scope.pop("restaurant_id"),\n                    "question": row["pergunta"],\n                    "filters": scope,\n                    "k": 8,\n                }\n                with httpx.Client(timeout=75) as client:\n                    if barrier:\n                        barrier.wait(timeout=10)\n                    started = perf_counter()\n                    try:\n                        reply = client.post(endpoint + "/ask", json=body)\n                        record = {"http_status": reply.status_code, "response": reply.json()}\n                    except (httpx.HTTPError, ValueError) as exc:\n                        record = {"client_error": type(exc).__name__}\n                    append(\n                        path,\n                        event="http",\n                        group=group,\n                        id=identifier,\n                        started=started,\n                        seconds=perf_counter() - started,\n                        **record,\n                    )\n\n            for identifier in ("q22", "q23"):\n                request(identifier, "warmup")\n            for number in (*range(1, 13), *range(14, 22)):\n                request(f"q{number:02}", "sequential")\n            barrier = threading.Barrier(2)\n            with ThreadPoolExecutor(max_workers=2) as pool:\n                futures = [\n                    pool.submit(request, identifier, "concurrent", barrier)\n                    for identifier in ("q24", "q25")\n                ]\n                for future in futures:\n                    future.result()\n        finally:\n            server.terminate()\n            try:\n                server.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                server.kill()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=("serve", "measure"))\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    os.environ["CUDA_VISIBLE_DEVICES"] = "0"\n    if args.mode == "serve":\n        serve(args.output)\n    else:\n        args.output.touch(exist_ok=False)\n        measure(args.output)\n\n\nif __name__ == "__main__":\n    main()\n',  # noqa: E501
}.items():
    Path("scripts", name).write_text(source, encoding="utf-8")

In [ ]:
download = """
import json
from huggingface_hub import HfApi, snapshot_download
model = "mistralai/Mistral-7B-Instruct-v0.2"
revision = HfApi().model_info(model, token=False).sha
snapshot = snapshot_download(model, revision=revision, token=False,
    allow_patterns=["config.json", "generation_config.json", "*.safetensors",
                    "*.safetensors.index.json", "tokenizer*", "special_tokens_map.json",
                    "chat_template.jinja"])
print(json.dumps({"model": model, "revision": revision, "snapshot": snapshot}))
"""
snapshot_info = Path("/tmp/mistral-benchmark-snapshot.json")
with snapshot_info.open("w", encoding="utf-8") as info:
    subprocess.run(
        [sys.executable, "-c", download],
        stdout=info,
        check=True,
        timeout=max(1, deadline - monotonic()),
    )
model_info = json.loads(snapshot_info.read_text(encoding="utf-8").splitlines()[-1])
snapshot = Path(model_info.pop("snapshot"))
journal(event="model_snapshot", **model_info)

In [ ]:
def pack():
    archive = shutil.make_archive(str(working / "mistral-t4-results"), "zip", out)
    print("Artefato:", archive, flush=True)


def stage(name, command, cap=None):
    journal(event="stage_start", stage=name)
    try:
        run(command, cap=cap)
    except (subprocess.SubprocessError, TimeoutError) as exc:
        journal(event="stage_error", stage=name, error=type(exc).__name__)
    finally:
        pack()


for mode in ("nf4", "fp16"):
    stage(
        mode,
        [
            sys.executable,
            "scripts/benchmark_gpu_precision.py",
            mode,
            str(snapshot),
            str(out / f"{mode}.jsonl"),
        ],
        cap=900,
    )

In [ ]:
os.environ.update(
    CHROMA_PATH=str(Path("/tmp/mistral-t4-chroma")),
    LLM_MODEL_ID=str(snapshot),
    LLM_BACKEND="real",
    API_ALLOW_MODEL_DOWNLOAD="true",
    API_REQUEST_TIMEOUT_S="60",
    EMBEDDING_DEVICE="cpu",
    ASPECT_ENRICHMENT_ENABLED="false",
    ASPECT_FILTER_ENABLED="false",
)
stage("api_index", [sys.executable, "scripts/ingest.py"], cap=360)
if (Path("/tmp/mistral-t4-chroma") / "chroma.sqlite3").exists():
    stage(
        "api_http",
        [sys.executable, "scripts/benchmark_gpu_api.py", "measure", str(out / "api.jsonl")],
        cap=900,
    )
journal(event="session_finished", elapsed_seconds=BUDGET_MIN * 60 - (deadline - monotonic()))
pack()